# Bài 9
Đây là notebook chứa mã nguồn đầy đủ của bài 9.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import pulp
import pyomo.environ as pyo
from scipy.optimize import linprog, minimize, milp, LinearConstraint, Bounds
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize as pymoo_minimize

from src.data_loader import get_data


In [ ]:
def solve_bai09(data_dir=None, ai_adoption_rate=0.30, retraining_budget=15,
                transition_speed=0.5, new_job_multiplier=0.4):
    data = get_data(data_dir)
    sectors = data.sectors_names_vi.tolist()

    employment = data.sectors_employment.astype(float)
    automation_risk = data.sectors_automation_risk.astype(float)
    N = len(sectors)
    
    # 1. PuLP LP Model
    m = pulp.LpProblem('VN_Labor_AI', pulp.LpMaximize)
    x_AI = pulp.LpVariable.dicts('x_AI', range(N), lowBound=0, upBound=1)
    x_H  = pulp.LpVariable.dicts('x_H', range(N), lowBound=0, upBound=1)
    
    # Objective: Maximize total NetJob
    # jobs_lost = L * Risk * rate * x_AI
    # jobs_created = jobs_lost * multiplier
    # jobs_retrained = L * speed * x_H
    # NetJob = jobs_created + jobs_retrained - jobs_lost
    net_jobs = []
    for i in range(N):
        j_lost = employment[i] * automation_risk[i] * ai_adoption_rate * x_AI[i]
        j_created = j_lost * new_job_multiplier
        j_retrained = employment[i] * transition_speed * x_H[i]
        net_jobs.append(j_created + j_retrained - j_lost)
        
    m += pulp.lpSum(net_jobs)
    
    # Budget constraints
    m += pulp.lpSum(x_AI[i] for i in range(N)) <= 5.0  # assumed total AI allocation capacity
    m += pulp.lpSum(x_H[i] for i in range(N)) <= retraining_budget / 10.0  # scaled
    
    m.solve(pulp.PULP_CBC_CMD(msg=False))
    
    # Collect results
    alloc_AI = [pulp.value(x_AI[i]) for i in range(N)]
    alloc_H = [pulp.value(x_H[i]) for i in range(N)]
    
    jobs_lost_val = [employment[i] * automation_risk[i] * ai_adoption_rate * alloc_AI[i] for i in range(N)]
    jobs_retrained_val = [employment[i] * transition_speed * alloc_H[i] for i in range(N)]
    jobs_created_val = [jl * new_job_multiplier for jl in jobs_lost_val]
    net_jobs_val = [jc + jr - jl for jc, jr, jl in zip(jobs_created_val, jobs_retrained_val, jobs_lost_val)]

    total_net = sum(net_jobs_val)

    # 2. Tìm ngưỡng x_H2 cho ngành Chế biến chế tạo (index 1)
    # NetJob2 >= 0 when x_AI2 = 1.0
    # j_created2 + j_retrained2 - j_lost2 >= 0
    # j_retrained2 >= j_lost2 - j_created2
    # L2 * speed * x_H2 >= (1 - multiplier) * (L2 * Risk2 * rate * 1.0)
    # x_H2 >= (1 - multiplier) * Risk2 * rate / speed
    threshold_xH2 = (1 - new_job_multiplier) * automation_risk[1] * ai_adoption_rate / transition_speed

    # 3. Sankey data for vulnerable groups (sectors 0, 2, 3 -> Nông nghiệp, Xây dựng, Khai khoáng)
    vul_indices = [0, 2, 3]
    sankey_nodes = ["Lao động ngành 1,3,4", "Việc làm bị mất", "Việc làm mới (AI)", "Chuyển đổi nghề", "Thất nghiệp ròng"]
    
    v_lost = sum(jobs_lost_val[i] for i in vul_indices)
    v_created = sum(jobs_created_val[i] for i in vul_indices)
    v_retrained = sum(jobs_retrained_val[i] for i in vul_indices)
    v_net_loss = v_lost - v_created - v_retrained
    if v_net_loss < 0: v_net_loss = 0
    
    sankey_links = {
        'source': [0, 1, 1, 1],
        'target': [1, 2, 3, 4],
        'value': [v_lost, v_created, v_retrained, v_net_loss]
    }

    # 4. Ràng buộc mở rộng: Không ngành nào mất quá 5% lao động
    m2 = pulp.LpProblem('VN_Labor_AI_Ext', pulp.LpMaximize)
    x_AI2 = pulp.LpVariable.dicts('x_AI2', range(N), lowBound=0, upBound=1)
    x_H2  = pulp.LpVariable.dicts('x_H2', range(N), lowBound=0, upBound=1)
    
    net_jobs2 = []
    for i in range(N):
        j_lost = employment[i] * automation_risk[i] * ai_adoption_rate * x_AI2[i]
        j_created = j_lost * new_job_multiplier
        j_retrained = employment[i] * transition_speed * x_H2[i]
        net_jobs2.append(j_created + j_retrained - j_lost)
        # Displaced = j_lost - j_created - j_retrained <= 0.05 * L_i
        m2 += (j_lost - j_created - j_retrained) <= 0.05 * employment[i]
        
    m2 += pulp.lpSum(net_jobs2)
    m2 += pulp.lpSum(x_AI2[i] for i in range(N)) <= 5.0
    m2 += pulp.lpSum(x_H2[i] for i in range(N)) <= retraining_budget / 10.0
    
    m2.solve(pulp.PULP_CBC_CMD(msg=False))
    ext_feasible = m2.status == pulp.LpStatusOptimal
    
    sector_table = {
        sectors[i]: {
            'x_AI': round(alloc_AI[i], 3),
            'x_H':  round(alloc_H[i], 3),
            'employment':  round(float(employment[i]), 2),
            'jobs_lost':   round(float(jobs_lost_val[i]), 3),
            'jobs_created':round(float(jobs_created_val[i]), 3),
            'net':         round(float(net_jobs_val[i]), 3),
        }
        for i in range(N)
    }

    return {
        'sectors': sectors,
        'alloc_AI': alloc_AI,
        'alloc_H': alloc_H,
        'net_jobs': net_jobs_val,
        'total_net': total_net,
        'threshold_xH2': threshold_xH2,
        'sankey_nodes': sankey_nodes,
        'sankey_links': sankey_links,
        'ext_feasible': ext_feasible,
        'sector_table': sector_table,
    }

In [ ]:
if __name__ == '__main__':
    res = solve_bai09()
    # In ra một số key để kiểm tra
    if isinstance(res, dict):
        print(res.keys())